In [2]:
import pandas as pd
import numpy as np
from pathlib import Path 

frequency = 64
path = Path(f"data/raw/dreamt/data_{frequency}Hz")

COLS_TO_DROP = [
    "TIMESTAMP",
    "IBI",
    "Obstructive_Apnea",
    "Central_Apnea",
    "Hypopnea",
    "Multiple_Events",
]
nb_users_max = 10
X_all_patients = []
y_all_patients = []
patient_file_list = [f for f in path.iterdir() if f.is_file()]
for patient_id in range(nb_users_max):
    patient_file = patient_file_list.pop() 
    df = pd.read_csv(patient_file)
    df["Sleep_Stage"] = df["Sleep_Stage"].replace("P", "W")
    df = df.drop(
                columns=COLS_TO_DROP
            )
    df = df[df["Sleep_Stage"] != "Missing"]
    y_all_patients.append(df.Sleep_Stage.to_numpy())
    X_all_patients.append(df.drop(columns=["Sleep_Stage"]).to_numpy())
    df["patient_id"] = patient_id + 1 



In [3]:
WINDOWS_SEC = 30
FS = 64

window_samples = FS * WINDOWS_SEC

X_bvp_patients = []
X_acc_patients = []
X_eda_temp_patients = []
X_hr_patients = []
y_patients = []

for patient in range(len(X_all_patients)):
    X_bvp = []
    X_acc = []
    X_eda_temp = []
    X_hr = []
    y = []
    data = X_all_patients[patient]
    T = data.shape[0]
    n_windows = T // window_samples
    for i in range(n_windows):
        start = i * window_samples
        end = start + window_samples
        # 1920, 
        X_bvp.append(data[start:end,0])
        # 960
        X_acc.append(data[start:end:2, 1:4])
        # 120
        X_eda_temp.append(data[start:end:16, 4:6])
        # 30
        X_hr.append(data[start:end:64, 6])
        #1
        y.append(y_all_patients[patient][start])
    
    X_bvp_patients.append(np.stack(X_bvp))
    X_acc_patients.append(np.stack(X_acc))
    X_hr_patients.append(np.stack(X_hr))
    X_eda_temp_patients.append(np.stack(X_eda_temp))
    y_patients.append(np.array(y))
        


In [4]:
X_bvp_train = []
X_bvp_test = []

X_acc_train = []
X_acc_test = []

X_eda_temp_train = []
X_eda_temp_test = []

X_hr_train = []
X_hr_test = []

y_train = []
y_test = []

test_size = 0.2

for patient in range(len(X_bvp_patients)):
    dataset_len = len(X_bvp_patients[patient])   
    split = 1 - int(dataset_len * test_size)
    X_bvp_train.append(X_bvp_patients[patient][:split])
    X_bvp_test.append(X_bvp_patients[patient][split:])
                      

    X_acc_train.append(X_acc_patients[patient][:split])
    X_acc_test.append(X_acc_patients[patient][split:])

                     
    X_eda_temp_train.append(X_eda_temp_patients[patient][:split])
    X_eda_temp_test.append(X_eda_temp_patients[patient][split:])
                       
    
    X_hr_train.append(X_hr_patients[patient][:split])
    X_hr_test.append(X_hr_patients[patient][split:])
                      
    y_train.append(y_patients[patient][:split])
    y_test.append(y_patients[patient][split:])



X_bvp_train =np.concatenate(X_bvp_train)
X_bvp_test =np.concatenate(X_bvp_test)

X_acc_train =np.concatenate(X_acc_train)
X_acc_test =np.concatenate(X_acc_test)

X_eda_temp_train =np.concatenate(X_eda_temp_train)
X_eda_temp_test =np.concatenate(X_eda_temp_test)

X_hr_train =np.concatenate(X_hr_train)
X_hr_test =np.concatenate(X_hr_test)

y_train = np.concatenate(y_train)
y_test = np.concatenate(y_test)


print(X_bvp_train.shape)
X_bvp_train = np.expand_dims(X_bvp_train, axis=1)
print(f"here {X_eda_temp_train.shape}")
X_eda_temp_train = np.permute_dims(X_eda_temp_train, axes=[0,2,1])
print(X_acc_train.shape)
X_acc_train = np.permute_dims(X_acc_train, axes=[0,2,1])

(8349, 1920)
here (8349, 120, 2)
(8349, 960, 3)


In [5]:
X_bvp_test      = np.expand_dims(X_bvp_test, axis=1)
X_eda_temp_test = np.permute_dims(X_eda_temp_test, axes=[0,2,1])
X_acc_test      = np.permute_dims(X_acc_test, axes=[0,2,1])

In [6]:
from sklearn.preprocessing import LabelEncoder

lb = LabelEncoder()
y_train_encoded = lb.fit_transform(y_train)
y_test_encoded = lb.transform(y_test)

y_train_encoded

array([4, 4, 4, ..., 3, 3, 3], shape=(8349,))

In [7]:
lb.classes_

array(['N1', 'N2', 'N3', 'R', 'W'], dtype='<U2')

In [34]:
from typing import Any

import torch 
import torch.nn as nn 


class Residual(nn.Module):
    def __init__(self, input_size, hidden_size, kernel_size = 3, dilation_n = 3):
        super().__init__()
        self.blocks = nn.ModuleList()
        original_input_size = input_size
        for i in range(dilation_n): 
            dilation = 2 ** i 
            self.blocks.append(nn.Sequential(
            nn.ZeroPad1d(padding=((kernel_size -1) * dilation,0)),
            nn.Conv1d(in_channels=input_size, out_channels =hidden_size, kernel_size=kernel_size, 
                                dilation=dilation),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU()
            ))
            input_size = hidden_size

        self.drop_out = nn.Dropout1d(0.5)
        self.align = nn.Conv1d(original_input_size, hidden_size, 1)  \
                if original_input_size != hidden_size else nn.Identity()
 


    def forward(self, input):
        x = input 
        for block in self.blocks:
            x = block(x)

        x = self.drop_out(x)

        return self.align(input) + x 
    


class MultiTCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.bc_bvp = nn.BatchNorm1d(1)
        self.bc_acc = nn.BatchNorm1d(3)
        self.bc_temp = nn.BatchNorm1d(2)

        self.bvp_block = Residual(1, 64, dilation_n=6)
        self.acc_block = Residual(3, 64, dilation_n=5)
        self.eda_temp = Residual(2, 64, dilation_n=3)
        target_length = 120


        self.avg_bvp = nn.AvgPool1d(kernel_size=1920 // target_length)
        self.avg_acc = nn.AvgPool1d(kernel_size=960 // target_length)    
        self.avg_temp = nn.AvgPool1d(kernel_size=120 // target_length)   
        self.fc = nn.Sequential(
            nn.Linear(64 * target_length * 3, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 5),
        ) 

        self.init_weights()

    def init_weights(self):
        for layer in self.modules():
            if isinstance(layer, (nn.Conv1d, nn.Linear)):
                nn.init.kaiming_normal_(layer.weight)
                nn.init.zeros_(layer.bias)
            
    def forward(self, x_bvp, x_acc, x_eda_temp, x_hr):
        x_bvp  = self.bc_bvp(x_bvp)
        x_acc = self.bc_acc(x_acc)
        x_eda_temp = self.bc_temp(x_eda_temp)
        out_bvp = self.bvp_block(x_bvp) #32, 1920
        out_acc = self.acc_block(x_acc) #32, 960
        out_eda_temp = self.eda_temp(x_eda_temp) #32, 120

        out_bvp = self.avg_bvp(out_bvp) #32, 30
        out_acc = self.avg_acc(out_acc) #32, 30 
        out_eda_temp = self.avg_temp(out_eda_temp) #32, 30 
        merged = torch.cat([out_bvp, out_acc, out_eda_temp], dim=1).flatten(1)

        return self.fc(merged)

In [60]:
import torch 
import torch.nn as nn 

class ResBlock1d(nn.Module):
    def __init__(self, channels, kernel_size=3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, padding=kernel_size//2),
            nn.BatchNorm1d(channels),
            nn.ReLU(),
            nn.Conv1d(channels, channels, kernel_size, padding=kernel_size//2),
            nn.BatchNorm1d(channels),
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(x + self.block(x))


class MultiScaleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # input (B, 1, 1920)
        self.bvp_path = nn.Sequential(
            nn.BatchNorm1d(1),
            nn.Conv1d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),                                        # →480
            ResBlock1d(32),
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),                                        # →120
            ResBlock1d(64),
            nn.Conv1d(64, 128, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),                                        # →30
            ResBlock1d(128),
        )  # (B, 128, 30)

        # input (B, 3, 960)
        self.acc_path = nn.Sequential(
            nn.BatchNorm1d(3),
            nn.Conv1d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),                                        # →240
            ResBlock1d(32),
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(4),                                        # →30
            ResBlock1d(64),
        )  # (B, 64, 30)

        # input (B, 2, 120)
        self.eda_temp_path = nn.Sequential(
            nn.BatchNorm1d(2),
            nn.Conv1d(2, 16, kernel_size=3, stride=2, padding=1),  # →60
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),                                        # →30
        )  # (B, 16, 30) — too shallow for residuals

        self.fc = nn.Sequential(
            nn.Linear(128*30 + 64*30 + 16*30 + 30, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 5),
        )

        self.hr_bn = nn.BatchNorm1d(1)
        self.init_weights()

    def init_weights(self):
        for layer in self.modules():
            if isinstance(layer, (nn.Conv1d, nn.Linear)):
                if not isinstance(layer, nn.modules.lazy.LazyModuleMixin):
                    nn.init.kaiming_normal_(layer.weight)
                    nn.init.zeros_(layer.bias)
            elif isinstance(layer, nn.BatchNorm1d):
                nn.init.ones_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, x_bvp, x_acc, x_eda_temp, x_hr):
        x_hr = self.hr_bn(x_hr.unsqueeze(1)).squeeze(1)
        out_bvp      = self.bvp_path(x_bvp).flatten(1)
        out_acc      = self.acc_path(x_acc).flatten(1)
        out_eda_temp = self.eda_temp_path(x_eda_temp).flatten(1)
        merged = torch.cat([out_bvp, out_acc, out_eda_temp, x_hr.flatten(1)], dim=1)
        return self.fc(merged)

In [9]:
from torch.utils.data import Dataset, DataLoader
class DreamtDataset(Dataset):
    def __init__(self, X_bvp, X_acc, X_eda_temp, X_hr, y):
        super().__init__()
        self.X_bvp      = X_bvp
        self.X_acc      = X_acc
        self.X_eda_temp = X_eda_temp
        self.X_hr       = X_hr
        self.y          = y

    def __getitem__(self, index):
        return (
            self.X_bvp[index],
            self.X_acc[index],
            self.X_eda_temp[index],
            self.X_hr[index],
            self.y[index],
        )

    def __len__(self):
        return len(self.X_bvp)



X_bvp_train      = torch.FloatTensor(X_bvp_train)
X_acc_train      = torch.FloatTensor(X_acc_train)
X_eda_temp_train = torch.FloatTensor(X_eda_temp_train)
X_hr_train       = torch.FloatTensor(X_hr_train)
y_train_encoded  = torch.LongTensor(y_train_encoded)  

train_ds = DreamtDataset(X_bvp_train, X_acc_train, X_eda_temp_train, X_hr_train, y_train_encoded)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)

In [10]:
from sklearn.utils.class_weight import compute_class_weight

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

classes = np.unique(y_train_encoded.numpy())
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_encoded.numpy())
weights = torch.FloatTensor(weights).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights, reduction="sum")

In [41]:
from tqdm import tqdm



X_bvp_test       = torch.FloatTensor(X_bvp_test)
X_acc_test       = torch.FloatTensor(X_acc_test)
X_eda_temp_test  = torch.FloatTensor(X_eda_temp_test)
X_hr_test        = torch.FloatTensor(X_hr_test)
y_test_encoded   = torch.LongTensor(y_test_encoded)

test_ds = DreamtDataset(X_bvp_test, X_acc_test, X_eda_temp_test, X_hr_test, y_test_encoded)
test_dl = DataLoader(test_ds, batch_size=1024)


def train_model(model, train_dl, epochs, weights= None, lr=0.001, device=torch.device("cpu")):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    if not weights:
        criterion = nn.CrossEntropyLoss(reduction="sum")
    else: 
        criterion = nn.CrossEntropyLoss(weight=weights, reduction="sum")
    for epoch in tqdm(range(epochs)):
        model.train()
        empirical_risk = 0.0
        for x_bvp, x_acc, x_eda_temp, x_hr, y in train_dl:
            x_bvp = x_bvp.to(device)
            x_acc = x_acc.to(device)
            x_eda_temp = x_eda_temp.to(device)
            x_hr = x_hr.to(device)
            y = y.to(device)
            optimizer.zero_grad()
            pred = model(x_bvp, x_acc, x_eda_temp, x_hr)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
            empirical_risk += loss.item()

        empirical_risk /= len(train_dl.dataset)
        print("Train loss: %.3f" % (empirical_risk))
        if epoch % 2 == 0:
            y_true, y_pred = test_model(model, test_dl, DEVICE)

def test_model(model, test_dl, device=torch.device("cpu")):
    criterion = nn.CrossEntropyLoss(reduction="sum")
    model.eval()
    generalization_error = 0.0
    correct = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for x_bvp, x_acc, x_eda_temp, x_hr, y in test_dl:
            x_bvp = x_bvp.to(device)
            x_acc = x_acc.to(device)
            x_eda_temp = x_eda_temp.to(device)
            x_hr = x_hr.to(device)
            y = y.to(device)
            logits = model(x_bvp, x_acc, x_eda_temp, x_hr)
            loss = criterion(logits, y)
            pred = torch.argmax(logits, dim=1)
            correct += (pred == y).sum().item()
            generalization_error += loss.item()
            all_preds.append(pred.cpu())
            all_targets.append(y.cpu())

        y_pred = torch.cat(all_preds).numpy()
        y_true = torch.cat(all_targets).numpy()
        generalization_error /= len(test_dl.dataset)
        accuracy = correct / len(test_dl.dataset)
        print(
            "Generalization Error: %.3f, Accuracy %.3f"
            % (generalization_error, accuracy)
        )

    return y_true, y_pred


model = MultiTCN()
model.to(DEVICE)

train_model(model, train_dl, epochs=10,device = DEVICE)


  0%|          | 0/10 [00:00<?, ?it/s]

Train loss: 5.442


 10%|█         | 1/10 [00:15<02:19, 15.55s/it]

Generalization Error: 2.932, Accuracy 0.372


OutOfMemoryError: CUDA out of memory. Tried to allocate 30.00 MiB. GPU 0 has a total capacity of 3.63 GiB of which 15.25 MiB is free. Including non-PyTorch memory, this process has 3.61 GiB memory in use. Of the allocated memory 3.34 GiB is allocated by PyTorch, and 194.96 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [40]:
import gc

def free_cuda():
    gc.collect()
    torch.cuda.empty_cache()


free_cuda()
del model 

In [ ]:

X_bvp_test       = torch.FloatTensor(X_bvp_test)
X_acc_test       = torch.FloatTensor(X_acc_test)
X_eda_temp_test  = torch.FloatTensor(X_eda_temp_test)
X_hr_test        = torch.FloatTensor(X_hr_test)
y_test_encoded   = torch.LongTensor(y_test_encoded)

test_ds = DreamtDataset(X_bvp_test, X_acc_test, X_eda_temp_test, X_hr_test, y_test_encoded)
test_dl = DataLoader(test_ds, batch_size=1024)

y_true, y_pred = test_model(model, test_dl, DEVICE)


Generalization Error: 1.834, Accuracy 0.475


In [66]:
from sklearn.metrics import f1_score, classification_report

print(f1_score(y_true, y_pred, average="macro"))

print(classification_report(y_true, y_pred, target_names=["N1","N2","N3","R","W"]))

0.22193700716512818
              precision    recall  f1-score   support

          N1       0.00      0.00      0.00      2704
          N2       0.55      0.80      0.66     10072
          N3       0.00      0.00      0.00       130
           R       0.90      0.00      0.00      3679
           W       0.38      0.55      0.45      4452

    accuracy                           0.50     21037
   macro avg       0.37      0.27      0.22     21037
weighted avg       0.50      0.50      0.41     21037



/volatile/home/yb285618/sleep-stage-prediction/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/volatile/home/yb285618/sleep-stage-prediction/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/volatile/home/yb285618/sleep-stage-prediction/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control thi

In [22]:
np.argwhere(y_test == "N2").shape

(8037, 1)